# IDH v2 · Tumour masks and tumour features on Kaggle (GPU, ~40 min)

Runs the GPU-heavy part of [DinoV2_IDH](https://github.com/Fuzzonaut/DinoV2_IDH) v2:
1. download FLAIR / T1 / T1c / T2 and the expert tumour masks (~6 GB)
2. segment every patient with the pretrained MONAI BraTS model (no training)
3. tumour-focused DINOv2 features + mask descriptors, from the predicted masks and from the expert masks

**Settings (right panel):** Accelerator = GPU (T4 or P100), Internet = On.
Then **Save Version → Save & Run All (Commit)**. When it has finished, download
`idh_v2_features.zip` from the *Output* tab.

In [ ]:
!git clone -q -b v2-segmentation https://github.com/Fuzzonaut/DinoV2_IDH.git /tmp/DinoV2_IDH
%cd /tmp/DinoV2_IDH
!pip install -q "monai>=1.5"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Data

In [ ]:
!python prepare_data.py --data-dir /tmp/data

## 2 · Tumour masks (pretrained BraTS SegResNet, Dice against the expert masks is reported)

In [ ]:
!python segment_pretrained.py --cohort /tmp/data/cohort.csv --masks-dir /kaggle/working/pred_masks \
    --results-dir /kaggle/working/results_v2

## 3 · Tumour features (predicted masks = real pipeline; expert masks = upper bound)

In [ ]:
!python extract_tumour_features.py --cohort /tmp/data/cohort.csv --masks pred \
    --masks-dir /kaggle/working/pred_masks --out-dir /kaggle/working/features_tumour_pred

In [ ]:
!python extract_tumour_features.py --cohort /tmp/data/cohort.csv --masks gt \
    --out-dir /kaggle/working/features_tumour_gt

## 4 · Package the output

In [ ]:
%cd /kaggle/working
!zip -qr idh_v2_features.zip pred_masks results_v2 features_tumour_pred features_tumour_gt
!rm -rf pred_masks features_tumour_pred features_tumour_gt
!ls -la /kaggle/working